# Generate example FMATCH and SCENE-ID products

This notebook writes one example (synthetic) NetCDF file per **FMATCH** product
definition and then chains the two camera-family FMATCH examples through the real
**SCENE-ID** runners to also produce example **SCENE-ID-CAM** and
**SCENE-ID-CAM-CAMTIME** files.

It is the notebook form of the former
`scripts/generate_fmatch_example_products.py`, extended with the scene-ID step.

## Why this notebook exists

The footprint-matching (FMATCH) PSF-aggregation engine that will eventually
*compute* the real values in these products is not implemented yet &mdash; the
producer functions in `libera_utils/footprint_matching/product.py` are still
`NotImplementedError` stubs. In the meantime, downstream consumers (Scene ID, the
ADM binning algorithm, the Camera Cloud Fraction algorithm) need a concrete,
*shape-and-format-correct* file to develop and test against. This notebook produces
exactly that: for every FMATCH operational mode it writes a NetCDF file that fully
conforms to the mode's product-definition YAML
(`libera_utils/data/product_definitions/fmatch_*.yml`).

## What is real vs. synthetic

Every FMATCH variable is a **synthetic placeholder** except an optional L1B
geolocation pass-through (see section 4). The synthetic values are only guaranteed
to (1) match the dtype declared in the product definition, (2) fall inside the
variable's `valid_range` when one is declared (and inside a physically plausible
range chosen from `units` otherwise), and (3) use valid IGBP land-cover codes
(1&ndash;20) for the `igbp_*surface_type*` variables &mdash; which the real SCENE-ID
classifier requires to derive `surface_type`. That is all a consumer needs from an
*example* file: the structure, dimensions, dtypes, attributes and encoding are the
contract; the magnitudes are not.

The **SCENE-ID** files, by contrast, are produced by running the *real* scene
classifier (`FootprintData.identify_scenes`) over the FMATCH example inputs, so
their `surface_type` / `cloud_fraction` / `scene_id_*` / `scene_bin_*` columns are
genuinely computed from the (synthetic) FMATCH inputs &mdash; only the inputs are
placeholders.

## 1. Imports

The synthetic-data helpers below are example-file-specific, so they are defined in
this notebook. Everything they build on &mdash; the product definitions, the L1B
pass-through reader, and the NetCDF writer &mdash; is imported from the installed
`libera_utils` package (that is production code the operational runners share).

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import xarray as xr

from libera_utils.footprint_matching.camera_segmentation import segment_l1b_camera

# Production helpers reused verbatim (these are the same functions the runners call).
from libera_utils.footprint_matching.l1b_inputs import (
    FMATCH_RADIOMETER_TIME_COORDINATE,
)
from libera_utils.footprint_matching.product import (
    fmatch_time_variable,
    load_fmatch_definition,
)
from libera_utils.footprint_matching.types import OperationalMode
from libera_utils.io.netcdf import write_libera_data_product
from libera_utils.io.product_definition import LiberaVariableDefinition

# The valid IGBP land-cover code set, used to keep synthetic igbp_*surface_type* values in range
# so the SCENE-ID classifier's IGBP -> TRMM surface_type conversion (which underpins every scene
# set) does not reject them. See synthesize_variable in section 3.
from libera_utils.scene_identification.scene_id import IGBPSurfaceType

## 2. Configuration

Edit these to control the run. All are plain notebook variables (they replace the
old script's command-line flags).

In [ ]:
# --- Where to write the example files ---------------------------------------
# Anchor everything to the repository root so the notebook writes to the same
# <repo-root>/example_outputs directory regardless of the kernel's working
# directory. We locate the root by walking up from the current directory until we
# find pyproject.toml (the marker at the repo root), falling back to cwd.
def _find_repo_root(start: Path) -> Path:
    """Return the nearest ancestor of ``start`` containing pyproject.toml (or ``start``)."""
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start.resolve()


REPO_ROOT = _find_repo_root(Path.cwd())
OUTPUT_DIR = REPO_ROOT / "example_outputs"

# --- How much synthetic data to generate ------------------------------------
# Number of synthetic footprints per file. Kept small (a real L1B day has ~600k
# footprints) so the example files stay tiny and quick to write/read; an example
# only needs enough records to exercise the format. When an L1B file is supplied
# (below), this same value is the target subsample size for the radiometer-timed
# modes; set it to None there to keep every valid L1B footprint.
N_FOOTPRINTS = None

# --- Optional real geolocation pass-through ---------------------------------
# Point this at an L1B RAD-4CH NetCDF file to copy the genuine RADIOMETER_TIME,
# latitude/longitude and viewing angles into the radiometer-timed FMATCH products
# (FMATCH-CAM, FMATCH-IMAGER-FLASH, FMATCH-IMAGER); every other variable stays
# synthetic, and the two camera-timescale modes stay fully synthetic (their
# CAMERA_TIME axis is not described by the L1B radiometer timeline). Leave as None
# for the original fully-synthetic behavior. Example:
#   L1B_FILE = REPO_ROOT / "external_data" / "LIBERA_L1B_RAD-4CH_V0-5-6_V0-5-6_20251120T175950_20251120T180020_R26190195454.nc"
L1B_FILE: Path | None = None
L1B_FILE = Path(
    "/workspaces/libera_utils/external_data/LIBERA_L1B_RAD-4CH_V0-5-5_20280212T033945_20280212T052007_R26154153835.nc"
)

# --- Optional L1B camera pass-through for camera-timescale modes ----------
# Point this at an L1B Daily Camera NetCDF file to segment it into pseudo-footprints
# and extract genuine L1B values (geolocation, angles, radiance) for the camera-
# timescale FMATCH products (FMATCH-CAM-CAMTIME, FMATCH-IMAGER-CAMTIME). The camera
# segmentation module creates radiometer-sized blocks from the camera pixel grid and
# reduces each block to a pseudo-footprint. When this is set, the camera-timescale
# modes use real (post-segmentation) L1B data instead of synthetic values, while all
# radiometer-timescale modes remain unaffected. Leave as None to keep camera-timescale
# modes fully synthetic. Example:
#   L1B_CAM_FILE = REPO_ROOT / "external_data" / "LIBERA_L1B_CAM_V0-2-6RC1_20280212T034136_20280212T034359_R26161155610.nc"
L1B_CAM_FILE: Path | None = None
L1B_CAM_FILE = Path(
    "/workspaces/libera_utils/external_data/LIBERA_L1B_CAM_V0-2-6RC1_20280212T034136_20280212T034359_R26161155610.nc"
)

In [3]:
# --- Synthetic timeline anchor ----------------------------------------------
# Time anchor for the synthetic footprint timeline. Borrowed from the example L1B
# RAD-4CH file (which starts 2028-02-12T03:39:45) so the example timestamps look
# like a real observation day. The radiometer samples at 100 Hz (10 ms cadence).
# These only drive the start/end time embedded in the generated filename; the
# instant values are not otherwise meaningful.
BASE_TIME = np.datetime64("2028-02-12T03:39:45.000000000", "ns")
SAMPLE_CADENCE = np.timedelta64(10_000_000, "ns")  # 10 ms == 100 Hz

# A single deterministic RNG so re-running the notebook reproduces identical files,
# which makes the examples stable to diff and safe to regenerate.
RNG = np.random.default_rng(seed=20280212)

# Physically-plausible fallback ranges keyed by the variable's declared ``units``.
# Used ONLY for variables that do not declare a ``valid_range``. The goal is
# realism, not correctness: e.g. a spacecraft altitude in meters should look like
# ~800 km, not ~0.5. Variables that DO declare a valid_range always use that range.
UNITS_FALLBACK_RANGE: dict[str, tuple[float, float]] = {
    "meters": (7.0e5, 8.5e5),  # spacecraft altitude above the ellipsoid (~LEO)
    "km": (0.0, 20.0),  # e.g. cloud heights
    "hPa": (100.0, 1013.0),  # atmospheric pressure (cloud-top .. surface)
    "m/s": (-20.0, 20.0),  # wind components can be negative
    "K": (200.0, 320.0),  # brightness/temperature-like quantities
    "g/m^2": (0.0, 500.0),  # column water paths
    "percent": (0.0, 100.0),  # concentrations/fractions in percent
    "1": (0.0, 1.0),  # dimensionless fractions / BRDF kernel weights
}
# Last-resort range for a dimensionless float variable with no usable hint.
UNITLESS_DEFAULT_RANGE: tuple[float, float] = (0.0, 1.0)
# Last-resort inclusive range for an integer variable with no declared range.
INTEGER_DEFAULT_RANGE: tuple[int, int] = (0, 10)

print(f"Repository root : {REPO_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"L1B pass-through: {'ON  -> ' + str(L1B_FILE) if L1B_FILE else 'OFF (fully synthetic)'}")

Repository root : /workspaces/libera_utils
Output directory: /workspaces/libera_utils/example_outputs
L1B pass-through: ON  -> /workspaces/libera_utils/external_data/LIBERA_L1B_RAD-4CH_V0-5-5_20280212T033945_20280212T052007_R26154153835.nc


## Camera Segmentation Pass-Through

The notebook can now optionally use real L1B camera data to generate example products for
the camera-timescale FMATCH modes (`FMATCH-CAM-CAMTIME`, `FMATCH-IMAGER-CAMTIME`).

When `L1B_CAM_FILE` is configured (section 2), the notebook:

1. **Loads the L1B Daily Camera NetCDF file** — a 3-D grid of 2048 × 2048 pixels per camera frame
2. **Segments each frame into pseudo-footprints** — using the camera segmentation module to divide
   the pixel grid into radiometer-sized blocks
3. **Extracts L1B values at each pseudo-footprint**:
   - **Geolocation/geometry**: center-pixel latitude, longitude, altitude, and viewing angles
   - **Pixel block identifiers**: center pixel indices and block boundaries (for provenance)
   - **Bounding box**: geographic extent of the block's corner pixels
   - **Quality flags**: segmentation flags (partial coverage, center pixel substitution)
   - **Radiance**: mean of valid pixels within each block

The resulting example products are "real-like": the geolocation, geometry, and radiance values
come directly from L1B, while all other variables (cloud fraction, PSF coverage, etc.) are
still synthetic placeholders. This gives downstream consumers concrete files that match the
real data structure without needing working implementations of the footprint-matching algorithms.

## 3. Synthetic-data helpers

These build the per-record arrays for the time/record axis and for every data
variable. Every FMATCH product is "SSF-style": a flat list of per-footprint records.
The **radiometer-timescale** modes index that list on `RADIOMETER_TIME`, so every
variable is 1-D along it. The **camera-timescale** modes index it on the `FOOTPRINT`
record axis, carrying `CAMERA_TIME`, `camera_pixel_x`, and `camera_pixel_y` as
*coordinates* &mdash; the two `camera_pixel_*` ranges being 2-D on
`[FOOTPRINT, CAMERA_PIXEL_BOUNDS]` (an inclusive `(min, max)` pixel-index pair).
Generation is fully generic &mdash; it introspects the loaded product definition's
variables *and* coordinates rather than hard-coding any list, so it stays correct as
the YAMLs evolve.

In [4]:
def value_range_for_variable(var_def: LiberaVariableDefinition) -> tuple[float, float]:
    """Pick a (low, high) range to draw synthetic values from for one variable.

    Preference order, most authoritative first:
      1. The variable's declared ``valid_range`` (honoring it guarantees in-range values).
      2. A physically-plausible range chosen from ``units`` so values look like the real quantity.
      3. A generic dimensionless fallback.
    """
    attrs = var_def.attributes
    # 1. Honor an explicit valid_range when the definition declares one.
    valid_range = attrs.get("valid_range")
    if valid_range is not None:
        return float(valid_range[0]), float(valid_range[1])

    # 2. Otherwise, fall back to a realistic range based on the declared units.
    units = attrs.get("units")
    if units in UNITS_FALLBACK_RANGE:
        return UNITS_FALLBACK_RANGE[units]

    # 3. No range and no recognized units: safe dimensionless default for floats;
    #    integer columns are handled separately in synthesize_variable.
    return UNITLESS_DEFAULT_RANGE


# IGBP surface-type codes are the discrete set defined by IGBPSurfaceType (1..20). The product
# definitions declare the igbp_*surface_type* variables as plain int16 with no valid_range, so the
# generic integer fallback in synthesize_variable would emit 0 and cap at 10 -- both invalid IGBP
# codes. The SCENE-ID classifier converts igbp_MODIS_surface_type into a TRMM surface_type (an
# input that underpins every scene set) and raises on any code outside 1..20, so synthetic IGBP
# codes must be drawn from the real code range.
IGBP_SURFACE_TYPE_RANGE: tuple[int, int] = (
    min(s.value for s in IGBPSurfaceType),
    max(s.value for s in IGBPSurfaceType),
)


def _is_igbp_surface_type(var_name: str) -> bool:
    """True for the IGBP land-cover *code* variables (igbp_..._surface_type[_primary|_secondary|...])."""
    return var_name.startswith("igbp") and "surface_type" in var_name


def synthesize_variable(var_def: LiberaVariableDefinition, n: int, var_name: str = "") -> np.ndarray:
    """Create a length-``n`` synthetic array for one (non-time) variable, matching its dtype."""
    dtype = np.dtype(var_def.dtype)

    # Domain rule: IGBP surface-type codes must be valid IGBPSurfaceType values (1..20) or the
    # SCENE-ID IGBP->TRMM surface_type conversion raises. The definitions declare no valid_range
    # for them, so pin the draw here before the generic integer fallback can emit 0 / cap at 10.
    if _is_igbp_surface_type(var_name):
        low, high = IGBP_SURFACE_TYPE_RANGE
        return RNG.integers(low, high + 1, size=n).astype(dtype)

    low, high = value_range_for_variable(var_def)

    if np.issubdtype(dtype, np.integer):
        # Integers (e.g. q_flags int64): draw inclusive of both bounds. If no range was declared,
        # value_range_for_variable returns the float UNITLESS default, so substitute the integer
        # default instead.
        if (
            var_def.attributes.get("valid_range") is None
            and var_def.attributes.get("units") not in UNITS_FALLBACK_RANGE
        ):
            low, high = INTEGER_DEFAULT_RANGE
        # randint's high is exclusive, hence +1 to make the declared max attainable.
        values = RNG.integers(int(low), int(high) + 1, size=n)
    else:
        # Floats: uniform across the chosen range is plenty for an example file.
        values = RNG.uniform(low, high, size=n)

    # Cast to the exact declared dtype so the write-time conformance check (which
    # compares dtype strings) passes without an auto-cast warning.
    return values.astype(dtype)


# The camera pixel grid is 2048 x 2048, so valid camera pixel indices are [0, CAMERA_PIXEL_MAX_INDEX].
# Used only to synthesize the camera_pixel_x/y range coordinates when no L1B camera file is supplied.
CAMERA_PIXEL_MAX_INDEX = 2047


def synthesize_coordinate(coord_def: LiberaVariableDefinition, n: int, coord_name: str = "") -> np.ndarray:
    """Create a length-``n`` synthetic array for one non-time coordinate, matching its dtype and shape.

    The leading (record) dimension is the ``FOOTPRINT`` axis of length ``n``. The only
    coordinates that carry a second dimension are the camera pixel-range coordinates
    (``camera_pixel_x`` / ``camera_pixel_y``), which are 2-D on
    ``[FOOTPRINT, CAMERA_PIXEL_BOUNDS]`` and hold an inclusive ``(min, max)`` pixel-index
    pair (``CAMERA_PIXEL_BOUNDS`` has size 2 per ``libera_dimensions.yml``). A purely 1-D
    coordinate falls through to :func:`synthesize_variable`.
    """
    # 1-D coordinates synthesize exactly like a data variable.
    if len(coord_def.dimensions) <= 1:
        return synthesize_variable(coord_def, n, coord_name)

    # 2-D camera pixel-range coordinate: draw an inclusive (min, max) pair per footprint.
    # Pick the lower bound in [0, MAX], then the upper bound in [low, MAX], so every pair
    # satisfies min <= max and stays on the 2048 x 2048 camera grid.
    dtype = np.dtype(coord_def.dtype)
    low = RNG.integers(0, CAMERA_PIXEL_MAX_INDEX + 1, size=n)
    high = RNG.integers(low, CAMERA_PIXEL_MAX_INDEX + 1)
    return np.stack([low, high], axis=1).astype(dtype)


def synthesize_time_coordinate(n: int) -> np.ndarray:
    """Create the monotonically increasing per-footprint datetime64[ns] time coordinate.

    ``write_libera_data_product`` requires the time variable to be datetime64 and uses
    its first/last elements to stamp the filename's start/end times, so it must be sorted.
    """
    offsets = np.arange(n, dtype="int64")
    return BASE_TIME + offsets * SAMPLE_CADENCE

In [5]:
def extract_camera_passthrough(
    l1b_cam_file: Path,
) -> dict[str, np.ndarray]:
    """Load an L1B Camera file, segment it, and extract L1B values for each pseudo-footprint.

    Uses the camera segmentation module to divide camera images into radiometer-sized
    pseudo-footprints, then extracts real L1B values (geolocation, angles, radiance) at
    each pseudo-footprint's center pixel and bbox, along with camera-specific identifiers
    (pixel blocks, quality flags, bounding boxes).

    Parameters
    ----------
    l1b_cam_file : Path
        Path to an L1B Daily Camera NetCDF file.

    Returns
    -------
    dict[str, np.ndarray]
        Arrays for CAMERA_TIME, geolocation (latitude, longitude, altitude), viewing
        geometry (solar_zenith_angle, viewing_zenith_angle, relative_azimuth_angle),
        camera block provenance (center_pixel_x, center_pixel_y boresight pixel, plus the
        camera_pixel_x/camera_pixel_y inclusive (min, max) index ranges), pseudo-footprint
        bounding box (psf_bbox_lat_min, psf_bbox_lat_max, psf_bbox_lon_min,
        psf_bbox_lon_max), quality flags (q_flags), and radiance (aggregated over block).
    """
    print(f"Loading L1B camera file: {l1b_cam_file}")
    with xr.open_dataset(l1b_cam_file) as l1b_cam:
        # Pre-load radiance data once for efficient access
        radiance_3d = np.asarray(l1b_cam["Radiance"].values, dtype=float)
        times_in_file = l1b_cam["CAMERA_TIME"].values

        # Segment into pseudo-footprints
        pseudo_footprints = segment_l1b_camera(l1b_cam)
        n_footprints = len(pseudo_footprints)
        print(f"  Segmented into {n_footprints} pseudo-footprints")

        # Pre-allocate arrays for each output variable
        times = np.empty(n_footprints, dtype="datetime64[ns]")
        lats = np.empty(n_footprints, dtype=np.float32)
        lons = np.empty(n_footprints, dtype=np.float32)
        alts = np.empty(n_footprints, dtype=np.float32)
        szas = np.empty(n_footprints, dtype=np.float32)
        vzas = np.empty(n_footprints, dtype=np.float32)
        raas = np.empty(n_footprints, dtype=np.float32)
        radiances = np.empty(n_footprints, dtype=np.float32)

        # Camera-specific identifiers (int32 to match product definition)
        center_x = np.empty(n_footprints, dtype=np.int32)
        center_y = np.empty(n_footprints, dtype=np.int32)
        # Inclusive (min, max) pixel-index ranges on the size-2 CAMERA_PIXEL_BOUNDS axis.
        camera_pixel_x = np.empty((n_footprints, 2), dtype=np.int32)
        camera_pixel_y = np.empty((n_footprints, 2), dtype=np.int32)

        # Bounding box coordinates
        bbox_lat_min = np.empty(n_footprints, dtype=np.float32)
        bbox_lat_max = np.empty(n_footprints, dtype=np.float32)
        bbox_lon_min = np.empty(n_footprints, dtype=np.float32)
        bbox_lon_max = np.empty(n_footprints, dtype=np.float32)

        # Quality flags (int32 to match product definition)
        q_flags = np.empty(n_footprints, dtype=np.int32)

        # Extract L1B values at each pseudo-footprint's center pixel
        for i, pf in enumerate(pseudo_footprints):
            times[i] = pf.time
            lats[i] = pf.latitude
            lons[i] = pf.longitude
            alts[i] = pf.altitude
            szas[i] = pf.solar_zenith_angle
            vzas[i] = pf.viewing_zenith_angle
            raas[i] = pf.relative_azimuth_angle

            # Camera pixel block identifiers
            center_x[i] = pf.center_ix
            center_y[i] = pf.center_iy
            # slice_x/slice_y are half-open [start, stop); the inclusive max is stop - 1.
            camera_pixel_x[i] = (pf.slice_x.start, pf.slice_x.stop - 1)
            camera_pixel_y[i] = (pf.slice_y.start, pf.slice_y.stop - 1)

            # Bounding box (lat/lon extent)
            bbox_lat_min[i] = pf.bbox.lat_min
            bbox_lat_max[i] = pf.bbox.lat_max
            bbox_lon_min[i] = pf.bbox.lon_min
            bbox_lon_max[i] = pf.bbox.lon_max

            # Quality flags (from camera segmentation)
            q_flags[i] = int(pf.q_flags)

            # Radiance: mean of pixels in the block (simple aggregation strategy).
            # Find the image index that matches this pseudo-footprint's time.
            t_idx = np.where(times_in_file == pf.time)[0][0]
            radiance_2d = radiance_3d[t_idx, :, :]
            block_radiance = radiance_2d[pf.slice_x, pf.slice_y]
            # Mean over valid pixels in the block (skip fill values and negatives)
            valid_mask = np.isfinite(block_radiance) & (block_radiance > 0)
            if np.any(valid_mask):
                radiances[i] = np.mean(block_radiance[valid_mask])
            else:
                radiances[i] = np.nan

    return {
        "CAMERA_TIME": times,
        "latitude": lats,
        "longitude": lons,
        "altitude": alts,
        "solar_zenith_angle": szas,
        "viewing_zenith_angle": vzas,
        "relative_azimuth_angle": raas,
        "radiance": radiances,
        "center_pixel_x": center_x,
        "center_pixel_y": center_y,
        "camera_pixel_x": camera_pixel_x,
        "camera_pixel_y": camera_pixel_y,
        "psf_bbox_lat_min": bbox_lat_min,
        "psf_bbox_lat_max": bbox_lat_max,
        "psf_bbox_lon_min": bbox_lon_min,
        "psf_bbox_lon_max": bbox_lon_max,
        "q_flags": q_flags,
    }

In [ ]:
def load_l1b_passthrough(l1b_file: Path, n_footprints: int | None) -> dict[str, np.ndarray]:
    """Read the real per-footprint L1B inputs, optionally subsampled for an example file.

    Reads all L1B radiometer inputs WITHOUT filtering for non-finite values. When
    ``n_footprints`` is given (and smaller than the available pool) we take an
    evenly-spaced subsample across the whole day (via linspace integer indices) so
    the example still spans the real time/space range.
    """
    # Map of FMATCH variable name -> L1B RAD-4CH variable name for pass-through variables.
    L1B_PASSTHROUGH_VARIABLES = {
        "latitude": "Latitude",
        "longitude": "Longitude",
        "solar_zenith_angle": "Solar_Zenith_Surface",
        "viewing_zenith_angle": "Viewing_Zenith_Surface",
        "relative_azimuth_angle": "Relative_Azimuth_Surface",
    }
    L1B_TIME_VARIABLE = "radiometer_time"

    # Open and read all L1B data without filtering
    with xr.open_dataset(l1b_file) as l1b:
        radiometer_time = l1b[L1B_TIME_VARIABLE].values
        passthrough = {
            fmatch_name: l1b[l1b_name].values.astype(np.float32)
            for fmatch_name, l1b_name in L1B_PASSTHROUGH_VARIABLES.items()
        }

    n_available = radiometer_time.size
    if n_footprints is None or n_footprints >= n_available:
        # Keep all footprints (including those with non-finite values)
        result = {FMATCH_RADIOMETER_TIME_COORDINATE: radiometer_time.astype("datetime64[ns]")}
        result.update(passthrough)
        return result

    # Subsample if requested: linspace with endpoint=True picks indices spread from
    # the first to the last row, preserving the full day's time/geographic span.
    indices = np.linspace(0, n_available - 1, num=n_footprints, dtype="int64")
    result = {FMATCH_RADIOMETER_TIME_COORDINATE: radiometer_time[indices].astype("datetime64[ns]")}
    result.update({name: values[indices] for name, values in passthrough.items()})
    return result


def build_synthetic_data(
    mode: OperationalMode,
    n_footprints: int | None,
    l1b_passthrough: dict[str, np.ndarray] | None = None,
    camera_passthrough: dict[str, np.ndarray] | None = None,
) -> tuple[dict[str, np.ndarray], str]:
    """Build the full {name: array} data dict for one operational mode.

    Introspects the loaded product definition so the variable set is always derived from
    the YAML, never hard-coded. When ``l1b_passthrough`` is supplied AND the mode is
    radiometer-timed, the RADIOMETER_TIME coordinate and every pass-through variable
    (lat/lon + viewing angles) are copied from the real L1B radiometer data; every other
    variable is synthesized. When ``camera_passthrough`` is supplied AND the mode is
    camera-timed, the CAMERA_TIME coordinate, the camera_pixel_x/y range coordinates,
    the geolocation/geometry variables, and the camera-specific identifiers are copied
    from the segmented L1B camera data; every other variable/coordinate is synthesized.
    """
    definition = load_fmatch_definition(mode)
    time_variable = fmatch_time_variable(mode)

    # Pass-through applies only when we have the matching L1B data AND the mode's
    # time axis matches.
    use_l1b_radiometer = l1b_passthrough is not None and time_variable == "RADIOMETER_TIME"
    use_l1b_camera = camera_passthrough is not None and time_variable == "CAMERA_TIME"

    data: dict[str, np.ndarray] = {}

    if use_l1b_radiometer:
        # Copy every real pass-through variable (time + lat/lon + viewing angles) straight
        # in from radiometer L1B.
        data.update(l1b_passthrough)
        n = data[time_variable].size
    elif use_l1b_camera:
        # Copy the time variable first (it's the coordinate, may not be in definition.variables)
        data[time_variable] = camera_passthrough[time_variable]

        # Copy all other available camera passthrough fields. This includes geolocation,
        # geometry, camera-specific identifiers, and bounding boxes. Any key in
        # camera_passthrough (other than time) that the definition declares -- as a data
        # variable OR as a coordinate -- gets copied; the rest (e.g. radiance, which no
        # camtime product declares) are skipped. camera_pixel_x/y are declared as 2-D
        # COORDINATES on [FOOTPRINT, CAMERA_PIXEL_BOUNDS], so they must be accepted here as
        # well; dropping them left the written file missing required coordinates.
        for key in camera_passthrough:
            if key == time_variable:
                continue  # Already handled above
            if key in definition.variables or key in definition.coordinates:
                data[key] = camera_passthrough[key]

        n = data[time_variable].size
    else:
        # Synthetic path: build the single time/record coordinate from the 100 Hz cadence
        # anchor. Use provided count, fallback to N_FOOTPRINTS, then to default for synthetic modes.
        n = n_footprints if n_footprints is not None else (N_FOOTPRINTS or 1000)
        data[time_variable] = synthesize_time_coordinate(n)

    # Every data variable hangs on the record dimension (RADIOMETER_TIME for radiometer
    # modes, FOOTPRINT for camera modes); synthesize each, skipping any variable already
    # populated from L1B pass-through above. The name is passed so domain rules (e.g. valid
    # IGBP surface-type codes) can apply.
    for var_name, var_def in definition.variables.items():
        if var_name in data:
            continue
        data[var_name] = synthesize_variable(var_def, n, var_name)

    # Synthesize any non-time coordinate the definition declares that pass-through did not
    # supply. For camera-timescale modes these are the 2-D camera_pixel_x/y range coordinates
    # (on [FOOTPRINT, CAMERA_PIXEL_BOUNDS]); the time coordinate is always populated above and
    # is skipped by the `in data` guard. Radiometer modes declare no such coordinate, so this
    # loop is a no-op for them.
    for coord_name, coord_def in definition.coordinates.items():
        if coord_name in data:
            continue
        data[coord_name] = synthesize_coordinate(coord_def, n, coord_name)

    return data, time_variable


def generate_example_product(
    mode: OperationalMode,
    output_dir: Path,
    n_footprints: int | None,
    l1b_passthrough: dict[str, np.ndarray] | None = None,
    camera_passthrough: dict[str, np.ndarray] | None = None,
) -> Path:
    """Generate and write one example NetCDF file for a single operational mode."""
    output_dir.mkdir(parents=True, exist_ok=True)
    definition = load_fmatch_definition(mode)
    data, time_variable = build_synthetic_data(mode, n_footprints, l1b_passthrough, camera_passthrough)

    # Supply the required *dynamic* product attributes that every FMATCH definition leaves
    # null (algorithm_version, input_files). These must be provided at write
    # time or conformance checking fails on null attributes.
    dynamic_product_attributes = {
        # Semantic version of the (synthetic) producer. Must match the semver regex.
        "algorithm_version": "0.1.0",
        # Provenance string. Real files list the L1B + ancillary inputs; this is an
        # example, so mark it synthetic to avoid implying real inputs.
        "input_files": "SYNTHETIC EXAMPLE - no real input files were used",
    }

    written = write_libera_data_product(
        definition,
        data,
        output_path=output_dir,
        time_variable=time_variable,
        dynamic_product_attributes=dynamic_product_attributes,
        strict=False,
    )
    return Path(str(written.path))

## 4. (Optional) load the real L1B pass-through inputs

Runs only when `L1B_FILE` is set (section 2). We load the pass-through arrays once
and reuse them for every radiometer-timed mode, so the large L1B file is opened and
filtered only once.

In [7]:
l1b_passthrough: dict[str, np.ndarray] | None = None
if L1B_FILE is not None:
    print(f"Reading L1B pass-through inputs from: {L1B_FILE}")
    l1b_passthrough = load_l1b_passthrough(L1B_FILE, N_FOOTPRINTS)
    n_used = l1b_passthrough[FMATCH_RADIOMETER_TIME_COORDINATE].size
    print(f"  Using {n_used} valid L1B footprints for the radiometer-timed modes.")
else:
    print("No L1B file configured -> radiometer-timed modes will be fully synthetic.")

camera_passthrough: dict[str, np.ndarray] | None = None
if L1B_CAM_FILE is not None:
    print(f"\nReading L1B camera pass-through inputs from: {L1B_CAM_FILE}")
    camera_passthrough = extract_camera_passthrough(L1B_CAM_FILE)
    n_cam = camera_passthrough["CAMERA_TIME"].size
    print(f"  Using {n_cam} camera pseudo-footprints for the camera-timed modes.")
else:
    print("\nNo L1B camera file configured -> camera-timed modes will be fully synthetic.")

Reading L1B pass-through inputs from: /workspaces/libera_utils/external_data/LIBERA_L1B_RAD-4CH_V0-5-5_20280212T033945_20280212T052007_R26154153835.nc


  Using 602200 valid L1B footprints for the radiometer-timed modes.

Reading L1B camera pass-through inputs from: /workspaces/libera_utils/external_data/LIBERA_L1B_CAM_V0-2-6RC1_20280212T034136_20280212T034359_R26161155610.nc
Loading L1B camera file: /workspaces/libera_utils/external_data/LIBERA_L1B_CAM_V0-2-6RC1_20280212T034136_20280212T034359_R26161155610.nc


  Segmented into 44055 pseudo-footprints


  Using 44055 camera pseudo-footprints for the camera-timed modes.


## 5. Generate the FMATCH example products

One file per operational mode. We keep each written path (keyed by mode) so section
6 can feed the two camera-family files straight into the scene-ID runners.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Writing FMATCH example products to: {OUTPUT_DIR}\n")

fmatch_outputs: dict[OperationalMode, Path] = {}
for mode in OperationalMode:
    path = generate_example_product(mode, OUTPUT_DIR, N_FOOTPRINTS, l1b_passthrough, camera_passthrough)
    fmatch_outputs[mode] = path
    # Flag which geolocation source each mode used.
    if camera_passthrough is not None and fmatch_time_variable(mode) == "CAMERA_TIME":
        source = "L1B camera (segmented)"
    elif l1b_passthrough is not None and fmatch_time_variable(mode) == "RADIOMETER_TIME":
        source = "L1B radiometer lat/lon/angles"
    else:
        source = "synthetic"
    print(f"  [{mode.value:>22}] ({source:>28}) -> {path.name}")

print("\nFMATCH example products done.")

## 6. Chain FMATCH &rarr; SCENE-ID products

The scene-identification runners take a FMATCH product as their operational input,
so we can feed the example files written above straight into them &mdash; this
exercises the *real* production read + classify + write path
(`FootprintData.from_fmatch_*` &rarr; `identify_scenes` &rarr;
`create_and_write_data_product_*`), not a bespoke demo mapping.

Two scene-ID products currently exist, one per camera-family FMATCH mode:

| Input FMATCH mode        | Reader                              | Output product        |
| ------------------------ | ----------------------------------- | --------------------- |
| `FMATCH-CAM`             | `from_fmatch_cam` (RADIOMETER_TIME) | `SCENE-ID-CAM`        |
| `FMATCH-CAM-CAMTIME`     | `from_fmatch_cam_camtime` (CAMERA_TIME) | `SCENE-ID-CAM-CAMTIME` |

The IMAGER-family FMATCH modes have no scene-ID runner yet, so they are not chained.
Both runners classify only the ERBE and unfiltering scene sets (TRMM's inputs are
unavailable in the FMATCH products). The camera-timescale product additionally
passes the FMATCH footprint *identifier* variables (camera pixel-block indices, PSF
bounding box, boresight geolocation) straight through.

In [9]:
from libera_utils.scene_identification.cam.scene_id_cam import (
    create_and_write_data_product_cam,
    run_scene_identification_cam,
)
from libera_utils.scene_identification.cam_camtime.scene_id_cam_camtime import (
    create_and_write_data_product_cam_camtime,
    run_scene_identification_cam_camtime,
)

# (FMATCH input mode, human label, classify fn, write fn) for each scene-ID product.
SCENE_ID_SPECS = [
    (
        OperationalMode.CAM,
        "SCENE-ID-CAM",
        run_scene_identification_cam,
        create_and_write_data_product_cam,
    ),
    (
        OperationalMode.CAM_CAMTIME,
        "SCENE-ID-CAM-CAMTIME",
        run_scene_identification_cam_camtime,
        create_and_write_data_product_cam_camtime,
    ),
]

print(f"Writing SCENE-ID example products to: {OUTPUT_DIR}\n")

scene_id_outputs: dict[str, Path] = {}
for mode, label, run_scene_identification, create_and_write_data_product in SCENE_ID_SPECS:
    fmatch_path = fmatch_outputs[mode]
    # run_* reads the FMATCH file and runs the ERBE + unfiltering classification;
    # create_and_write_* finalizes onto the product's time axis and writes the NetCDF.
    footprint_data = run_scene_identification(fmatch_path)
    written = create_and_write_data_product(footprint_data, fmatch_path.name, OUTPUT_DIR)
    scene_id_outputs[label] = Path(str(written.path))
    print(f"  [{label:>22}] from {fmatch_path.name}")
    print(f"  {'':>22}   -> {scene_id_outputs[label].name}")

print("\nSCENE-ID example products done.")

Writing SCENE-ID example products to: /workspaces/libera_utils/example_outputs



  [          SCENE-ID-CAM] from LIBERA_AUX_FMATCH-CAM_V0-1-0_20280212T033945_20280212T052007_R26224184847.nc
                           -> LIBERA_AUX_SCENE-ID-CAM_V0-1-0_20280212T033945_20280212T052007_R26224184847.nc
  [  SCENE-ID-CAM-CAMTIME] from LIBERA_AUX_FMATCH-CAM-CAMTIME_V0-1-0_20280212T034136_20280212T034359_R26224184847.nc
                           -> LIBERA_AUX_SCENE-ID-CAM-CAMTIME_V0-1-0_20280212T034136_20280212T034359_R26224184847.nc

SCENE-ID example products done.


## 7. Inspect an output

A quick look at one product to confirm the structure. The SCENE-ID-CAM file carries
the classification columns (`surface_type`, `cloud_fraction`, `scene_id_erbe`,
`scene_id_unfiltering`, and the `scene_bin_*` bounds) computed from the FMATCH inputs.

In [10]:
scene_id_cam_path = scene_id_outputs["SCENE-ID-CAM"]
with xr.open_dataset(scene_id_cam_path) as scene_id_cam:
    print(scene_id_cam)

<xarray.Dataset> Size: 60MB
Dimensions:                                           (RADIOMETER_TIME: 602200)
Coordinates:
  * RADIOMETER_TIME                                   (RADIOMETER_TIME) datetime64[ns] 5MB ...
Data variables: (12/29)
    igbp_surface_type                                 (RADIOMETER_TIME) uint8 602kB ...
    cloud_fraction                                    (RADIOMETER_TIME) float32 2MB ...
    solar_zenith_angle                                (RADIOMETER_TIME) float32 2MB ...
    viewing_zenith_angle                              (RADIOMETER_TIME) float32 2MB ...
    relative_azimuth_angle                            (RADIOMETER_TIME) float32 2MB ...
    surface_type                                      (RADIOMETER_TIME) uint8 602kB ...
    ...                                                ...
    scene_bin_unfiltering_solar_zenith_angle_max      (RADIOMETER_TIME) float32 2MB ...
    scene_bin_unfiltering_surface_type_min            (RADIOMETER_TIME) uint8 602kB .